# P02 · Resolution test (ADR 0001 round 3) + benchmark v0

Runs DINOv3 ViT-B/16 on the **development and secondary** events at two input resolutions
(native, and resampled to a common 0.3 m), pools every building (outline and 3 m dilated), then runs
`bench run`: leave-one-event-out probes, per-event AUROC and event-level comparisons.

* **Answers:** does 0.3 m beat native resolution (ADR 0001)? Does dilated pooling beat outline? It is also
  **benchmark v0** and gives the pilot spread of per-event gains used for the power check.
* **Exploratory only.** Before the 10 Jan freeze, confirmatory tiles are skipped and every verdict is labelled
  "exploratory reading, not a claim".
* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN` and `HF_TOKEN`.
  Needs `01_data/derived/buildings.parquet` from **p01**.
* **Time:** roughly 30–45 minutes (native is fast; 0.3 m makes Beirut and Marshall tiles 1.7–2× larger per side).
* **Outputs:** features in `02_models/features/<date>_dinov3_resolution/`, the run (scores, metrics,
  comparisons, `report.md`, timing) in this run's folder.

In [ ]:
NOTEBOOK_NAME = "p02_resolution_benchmark_v0"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU and Hugging Face token (DINOv3 is gated)
import os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])]
kept = use[use["excluded_reason"].isna()]
print(kept.groupby(["event", "role"]).agg(buildings=("instance_id", "size"), damaged=("damaged", "sum"),
                                         tiles=("tile_id", "nunique")).to_string())

In [ ]:
# Extract DINOv3 features at native and 0.3 m, pooled with outline and 3 m dilation
import datetime
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile, write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, resolution_comparisons, runs_table
RESOLUTIONS = [None, 0.3]
METHODS = ["outline", "dilated:3"]
enc = get_encoder("dinov3_vitb16")
feats, runs = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, METHODS, RESOLUTIONS,
                                     "/content/features", events, log=print)
timing = runs_table(runs)
display(timing)
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_dinov3_resolution"
os.makedirs(FEAT_DIR, exist_ok=True)
FEATS = f"{FEAT_DIR}/features.parquet"
write_features(feats, FEATS)   # sorted by setting, small row groups: bench run reads one setting at a time
timing.to_csv(f"{RUN}/timing.csv", index=False)
timing.to_csv(f"{FEAT_DIR}/timing.csv", index=False)
print(len(feats), "feature rows saved to", FEATS)

In [ ]:
# bench run: probes per held-out development event, per-event AUROC, event-level comparisons
# Free the extraction tables first: bench run reads the features back one setting at a time.
import gc
del feats, use, kept, B
gc.collect()
torch.cuda.empty_cache()
from IPython.display import Markdown
from damagesar_bench.run import run_benchmark
comps = resolution_comparisons(enc.name, RESOLUTIONS, METHODS)
out = run_benchmark([FEATS], comps, events, f"{RUN}/bench_run", n_boot=2000)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))

In [ ]:
# Compact view for ADR 0001 round 3: AUROC per event and setting, and the spread of the gains
m = pd.read_csv(f"{out}/event_metrics.csv")
print(m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "strength", "n_events", "estimate", "lower", "upper", "verdict"]].round(4).to_string(index=False))
import json, numpy as np
for _, r in c.iterrows():
    g = np.array(list(json.loads(r["gains"]).values()))
    print(f"{r['label']}: SD of per-event gains = {g.std(ddof=1):.4f}")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))